In [ ]:
checkpoint_path: str='./lightning_logs/Everything_TS=4/1488392/last.ckpt' # can be a glob pattern, in which case the last checkpoint is chosen
n_flow_through_times: int=2 # number of flow through times to simulate to determine long term behavior
n_sim_pred_samples: int=5 # number of samples to use for the simulation
n_intermediate_outputs: int=25 # number of intermediate outputs to use for the simulation
fast_dataloaders: bool=True # whether to use fast dataloaders (don't overwhelm memory with large models)
random_IC: bool=False # whether to use a random IC for the simulation
seed: int=0 # seed for RNG

# Setup:

## Log Commit:

In [ ]:
# track modified sub-modules
!git status

In [ ]:
!git diff HEAD -- *.py

In [ ]:
!git show | \head -n1

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt
import pytorch_lightning as L
import torch.nn.functional as F
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import scrapbook as sb
def glue_and_print(key, value):
    try: value=value.item()
    except: pass
    print(f'{key}={value}')
    sb.glue(key, value)

from utils import *
from lightning_utils import *
import JHTDB_sim_op
from JHTDB_sim_op import POU_NetSimulator, PPOU_NetSimulator

import model_agnostic_BNN # the script is now fully compatible with the current model
from model_agnostic_BNN import PredSamplingWrapper

L.seed_everything(seed)

In [ ]:
from glob import glob
try: checkpoint_path = sorted(glob(checkpoint_path))[-1] # alphabetical last is also most recent! (both in terms of job id and last > epoch=X save)
except IndexError: raise FileNotFoundError(f'No checkpoint found at {checkpoint_path}') from None
print(f'{checkpoint_path=}')
sb.glue('checkpoint_path', checkpoint_path)

In [ ]:
import warnings
if not checkpoint_path.endswith('last.ckpt'):
    warnings.warn(f'{checkpoint_path} does not end with "last.ckpt", this job was incomplete!')

## Load the Model:

In [ ]:
import sys
sys.path.append('./WNO/Version_2.0.0')
sys.path.append('./IUNFO-CHL')
from glob import glob

import utils
def load_model(path, device='cuda', **kwd_args):
    ''' Wraps up all the nonsense involved in loading an inference model properly into one function. '''
    paths = glob(path)
    assert len(paths)==1, f'found these files: {paths}, but expected to find exactly one.'
    path = paths[0]
    print(f'loading model from path: {path}')

    try:
        print('loading VI model...')
        model = PPOU_NetSimulator.load_from_checkpoint(path, **kwd_args)
        PredSamplingWrapper.wrap_VI_model(model)
    except Exception as e:
        try:
            print('loading deterministic model...')
            model = POU_NetSimulator.load_from_checkpoint(path, **kwd_args)
        except: raise

    model = model.to(device)
    model.eval()

    print(f'num model parameters: {utils.count_parameters(model):.5e}')

    # freeze everything
    for parameter in model.parameters():
        parameter.requires_grad=False
    print('done!')
    return model

# 'last.ckpt' comes after 'epoch*.ckpt' alphabetically (epoch*.ckpt is the "best long horizon model")
get_best_long_horizon_model = lambda ckpt_dir: sorted(glob(f'{ckpt_dir}/*.ckpt'))[0]

model = load_model(checkpoint_path)

### 3d Expert Partitions

In [ ]:
L.seed_everything(seed)
u0 = model.simulator.genIC().detach() # random IC
gating_weights, topk = model.gating_net(u0[None])
topk_sort_idx = torch.argsort(topk)
gating_weights, topk = gating_weights[:,topk_sort_idx], topk[topk_sort_idx]
gating_weights = gating_weights[0].cpu() # ditch batch dim & move to cpu
print(f'{gating_weights.shape=}')

In [ ]:
# add the zero expert weight
ones = torch.ones(1,*gating_weights.shape[1:], dtype=gating_weights.dtype, device=gating_weights.device)
zero_weight = ones - gating_weights.sum(axis=0)
gating_weights = torch.cat([zero_weight, gating_weights], axis=0)
print(f'{gating_weights.shape=}')

In [ ]:
from grid_figures import GridFigure
fig = GridFigure(f'3d Expert Partitions', y_title_vertical=False)

print('NOTE: expert #0 is also literally the "Zero Expert"')
for expert_i in range(gating_weights.shape[0]):
    fig.add_3d_row(gating_weights[expert_i], f'expert #{expert_i}', x_title_func=lambda t: f'z={t}',
                   img_getter=lambda array_3d, t: array_3d[:,:,t].T, time_samples=list(range(0,gating_weights.shape[-1],10)))
fig.show()

In [ ]:
# check if others are different? turns out the others are used a little bit
print('spatially averaged gating weights for each expert:', torch.vmap(torch.mean)(gating_weights).numpy())

## Data Loading:

In [ ]:
from JHTDB_data_loading import JHTDBDataModule, preload_dataset

# long horizon is explicit for legacy reasons to patch a previous bug
data_module = JHTDBDataModule.load_from_checkpoint(checkpoint_path, long_horizon=400, fast_dataloaders=fast_dataloaders)

In [ ]:
field_size = data_module.field_size
n_steps_total = 4000 // data_module.time_stride

# stride in "dataset timesteps" (i.e., after data_module.time_stride is applied)
intermediate_output_stride = max(1, n_steps_total // n_intermediate_outputs)
n_steps_total_intermediate = (n_steps_total - 1) // intermediate_output_stride + 1
time_stride_intermediate = data_module.time_stride * intermediate_output_stride

print(f'{n_steps_total=}')
print(f'{n_intermediate_outputs=}')
print(f'{intermediate_output_stride=}, {n_steps_total_intermediate=}, {time_stride_intermediate=}')

In [ ]:
# My temporary diagnostic utilities, TODO: DELETE ME
def compute_intermediate_output_stride(time_stride, n_intermediate_outputs):
    n_steps_total = 4000 // time_stride

    # stride in "dataset timesteps" (i.e., after data_module.time_stride is applied)
    intermediate_output_stride = max(1, n_steps_total // n_intermediate_outputs)
    n_steps_total_intermediate = (n_steps_total - 1) // intermediate_output_stride + 1
    total_intermediate_output_stride = time_stride * intermediate_output_stride

    print(f'{n_steps_total=}, {n_steps_total_intermediate=}')
    print(f'{total_intermediate_output_stride=}, {intermediate_output_stride=}')

def compute_EMA_samples(time_stride, intermediate_output_stride, EMA_steps=25):
    n_steps_total = 4000 // time_stride
    existing_steps = np.arange(4000)[::time_stride][::intermediate_output_stride]
    return existing_steps[np.linspace(0, len(existing_steps)-1, endpoint=True, num=EMA_steps, dtype=int)]

compute_EMA_samples(time_stride_intermediate, intermediate_output_stride)

In [ ]:
import numpy as np
def make_4d_sim_fig(sim_data, vel_comp_idx:int, prefix='', time_stride=time_stride_intermediate, 
                    num_z=6, vel_component_name = ['X','Y','Z'], show=True):
    from grid_figures import GridFigure
    fig = GridFigure(f'{prefix}3d Channel Flow: {vel_component_name[vel_comp_idx]} Velocity')
    sim_data = sim_data.cpu()
    for z in np.linspace(0, sim_data.shape[-2]-1, num=num_z, dtype=int):
        fig.add_3d_row(sim_data[vel_comp_idx,:,:,z], f'{z=}', x_title_func=lambda t: f't={t*time_stride}',
                       img_getter=lambda array_3d, t: array_3d[:,:,t].T)
    if show: fig.show()
    return fig

x, y = data_module.dataset[0]
xy = torch.cat([x[...,None], y],axis=-1) # recombine!
make_4d_sim_fig(xy, 0, time_stride=data_module.time_stride, num_z=field_size[-1]//8)

### Here we load the DNS dataset tensor:

In [ ]:
# load the full simulation tensor
real_channel_flow = data_module.load_full_dataset_field()

# strided view for comparisons/plots (keeps memory & compute down)
real_channel_flow_intermediate = real_channel_flow[..., ::intermediate_output_stride]

print('real_channel_flow (full):')
tshow(real_channel_flow) # show size in GBs
print(f'{real_channel_flow_intermediate.shape=}')
#print(f'real_channel_flow size: {np.prod(real_channel_flow.shape)*4/1e9} GB')

### Compute the Characteristic Time:
GOTCHA: only works for MSE, different errors require different models! \
For dataset (with MSE): `t_c=127.52619893227192` `C=0.0019192068442813793`

In [ ]:
class CharacteristicTimeMSEModel:
    def __init__(self, field_tensor, n_windows=2):
        field_tensor = field_tensor.detach()
        window_size = field_tensor.shape[-1] - n_windows + 1
        print(f'window_size=field_tensor.shape[-1]-n_windows+1={window_size}')
        # ^ verified to work: 1/21/26

        # basic u0 MSE (for plotting later)
        self.u0 = field_tensor[...,0]
        self.u0_MSE = torch.vmap(torch.mean)((field_tensor.moveaxis(-1, 0)-self.u0)**2)

        MSEs = [] # MSEs[i] is the MSE of the u0 from time i to i+window_size
        from tqdm import tqdm
        for i in tqdm(range(field_tensor.shape[-1]-window_size+1)):
            u0 = field_tensor[...,i]
            time_window = field_tensor.moveaxis(-1, 0)[i:i+window_size]
            MSEs.append(torch.vmap(torch.mean)((time_window-u0)**2))
        MSEs = torch.stack(MSEs, dim=0)

        import scipy.optimize as optimize
        t = np.arange(window_size)
        def exp_error_residual(x):
            C, t_c = x # this is what the array means
            pred_MSE = C*(1-np.exp(-t/t_c))
            return (abs(MSEs-pred_MSE)).mean().item()
        opt_result = optimize.shgo(exp_error_residual, bounds=((1e-16, 100), (1, 500)))
        #opt_result = optimize.minimize(exp_error_residual, x0=np.random.uniform(1, 250, size=2))
        self.asymptotic_MSE, self.characteristic_time = opt_result.x
        print(f'{opt_result=}')
        print(f'characteristic_time=t_c={self.characteristic_time}, asymptotic_MSE=C={self.asymptotic_MSE}')

    def predict_MSE(self, t):
        return self.asymptotic_MSE*(1-np.exp(-t/self.characteristic_time))

    def plot_MSE_vs_pred(self, other_model=None, other_model_label='other'):
        print(f'characteristic_time=t_c={self.characteristic_time}, asymptotic_MSE=C={self.asymptotic_MSE}')
        t = np.arange(self.u0_MSE.shape[0])
        if other_model:
            plt.plot(other_model.u0_MSE, label=other_model_label+'.u0_MSE')
            plt.plot(other_model.predict_MSE(t), label=other_model_label+'.prediction')
        plt.plot(self.u0_MSE, label='$MSE(u_0,u_t)$')
        plt.plot(self.predict_MSE(t), label='prediction')
        plt.axvline(self.characteristic_time, color='k', linestyle='--', label=f't_c={self.characteristic_time}')
        plt.axhline(self.asymptotic_MSE, color='k', linestyle=':', label=f'C={self.asymptotic_MSE}')
        plt.legend()
        plt.title('MSE of u0 vs time')
        plt.show()

In [ ]:
mse_model = CharacteristicTimeMSEModel(real_channel_flow)

In [ ]:
mse_model.plot_MSE_vs_pred()

In [ ]:
'''
NO_mse_model = CharacteristicTimeMSEModel(sim.flow_thru[0], n_windows=100)
NO_mse_model.plot_MSE_vs_pred()
mse_model.plot_MSE_vs_pred(other_model=NO_mse_model, other_model_label='NO_MSE')
''';

# Run Simulation:
This generalizes the case UQ and deterministic case (automatically taking epistemic samples if needed) 

In [ ]:
L.seed_everything(seed)
u0 = model.simulator.genIC(from_LES=True).detach() if random_IC else real_channel_flow[..., 0].to(model.device)
with torch.inference_mode(): # NOTE: to_cpu is only +2 secs per sample
    with PredSamplingWrapper.enable_sampling(n_sim_pred_samples, moments=False, verbose=True):
        sim_steps_raw = model(u0, n_steps=n_steps_total*n_flow_through_times, to_cpu=True, 
                              intermediate_output_stride=intermediate_output_stride)

In [ ]:
class Simulation:
    @classmethod
    def from_output(cls, sim_data, n_steps_total=n_steps_total):
        ''' Constructs a Simulation object from the output of the model.
        If the model outputs UQ, then the nested Simulation.uq Simulation will also contain simulation uncertainty.
        And the Simulation.uq.sample_moments attribute will contain the raw sample moments '''

        sim_uq = None # default
        if type(sim_data) in (tuple, list): # handle uq
            assert len(sim_data) == 2
            sim_samples, sim_samples_uq = sim_data # unpack

            # aggregate moments via law of total variance
            sim_data = sim_samples.mean(0)
            sim_data_uq = ((sim_samples_uq**2).mean(0) + sim_samples.var(0))**0.5
            sim_uq = cls(sim_data_uq, n_steps_total=n_steps_total,
                         sample_moments=(sim_samples, sim_samples_uq))

        return cls(sim_data, n_steps_total=n_steps_total, uq=sim_uq)

    def __init__(self, sim_data, n_steps_total=n_steps_total, uq=None, 
                 sample_moments: tuple[torch.Tensor, torch.Tensor]|None=None):
        self.full = sim_data
        n_flow_through_times = int(sim_data.shape[-1]/n_steps_total+0.5)
        self.flow_thru = [sim_data[...,n_steps_total*i:n_steps_total*(i+1)] 
                          for i in range(n_flow_through_times)]
        self.uq = uq # nested UQ Simulation
        self.sample_moments = sample_moments # raw sample moments (for flow stats)

sim = Simulation.from_output(sim_steps_raw, n_steps_total=n_steps_total_intermediate)

## Display Simulation:

In [ ]:
print(f'{u0.shape=}\n{real_channel_flow_intermediate.shape=}\n{sim.flow_thru[1].shape=}\n{sim.full.shape=}\n(uq variables have the same shape))')

In [ ]:
def create_all_simulation_figs(predicted_sim_steps, true_sim_steps=None, predicted_prefix='Learned ', true_prefix='DNS ',
                               time_stride=time_stride_intermediate, shared_color_bar=True):
    for i in range(3):
        if true_sim_steps is not None:
            print('#'*50 + f'\n{true_prefix}Simulation:')
            true_fig = make_4d_sim_fig(true_sim_steps, i, time_stride=time_stride, prefix=true_prefix)
        print('#'*50 + f'\n{predicted_prefix}Simulation:')
        fig = make_4d_sim_fig(predicted_sim_steps, i, time_stride=time_stride, prefix=predicted_prefix, show=False)
        if true_sim_steps is not None and shared_color_bar: fig.copy_color_bar(true_fig)
        fig.show()

create_all_simulation_figs(sim.flow_thru[0], real_channel_flow_intermediate)

## Display Simulation (Extrapolation Regime):

In [ ]:
if sim.uq:
    tshow(sim.uq.full)
    create_all_simulation_figs(sim.full, sim.uq.full*np.sqrt(2/np.pi), 
                               true_prefix='UQ ', shared_color_bar=False)
else: create_all_simulation_figs(sim.full)

In [ ]:
if sim.uq:
    vmean = torch.vmap(torch.mean)
    uq_over_time = vmean(sim.uq.full.swapaxes(0,-1))
    sb.glue('uq_over_time', uq_over_time.tolist())
    plt.plot(uq_over_time)
    plt.title('UQ vs time')
    plt.show()

# Ravi's Flow Statistics:

## Ravi's Energy Spectrum Equations:

"A good place to start is looking at the energy spectrum which describes how the energy is distributed across wavelengths.
Here’s how to get it:" <br>
![Ravi's FLow Stats Equations](attachment:524a632d-3843-4fc6-b640-edba1e83601e.png)

## Cross Correlation (checking for trivial advection):
Verified to work: turns out real_channel_flow is *exactly self consistent* with regards to xcor so the test is valid!

In [ ]:
def self_xcor(flow_data):
    ''' Takes the X-cross-correlation of the flow data with itself beginning vs ending of the flow.
    Assumes flow_data.shape==(3,Nx,Ny,Nz,times) '''
    f1 = flow_data[...,0]
    f2 = flow_data[...,-1]

    # * .mean(0,2,3) can be on the outside of the irfft but it is moved inside to make the irfft more efficient
    # * and iFFT(FFT(X).conj()*FFT(Y)) = cross_correlation(X,Y) (essentially template matching by sliding one across the other
    # * .conj() ensures similar signal maximizes the real component (x-iy)(x+iy)=x^2-(iy)^2=x^2+y^2 (i.e. is necessary to avoid the "flip" that happens in convolution)
    # * p1-p1.mean(1)[:,None] centers each signal like how you center variables when taking correlation (however no normalization is done)
    return np.fft.irfft((np.fft.rfft(f1-f1.mean(1)[:,None],axis=1).conj()*np.fft.rfft(f2-f2.mean(1)[:,None],axis=1)).mean((0,2,3)))

MSE = lambda pred, true: float(np.mean((np.asarray(pred) - np.asarray(true))**2))

def xcor_metrics(pred_xcor, true_xcor):
    metrics=np.array([(MSE(pred_xcor, true_xcor), pred_xcor.max()-true_xcor.max(), pred_xcor.argmax()-true_xcor.argmax())], 
              dtype=[('xcor_mse', 'f4'), ('xcor_peak_magnitude_delta', 'f4'), ('xcor_peak_loc_delta', 'f4')])
    return metrics

# NOTE: peaks indicate "template matches", strength of match is given by peak amplitude
# The rest of the signal is still useful for MSE but not necessary to aggregate
def cross_correlation_comparison(pred_flow, real_channel_flow, title='', plot=True):
    ''' This code assumes you have 2 numpy arrays loaded in memory:
    pred_samples with shape, (3,Nx,Ny,Nz,times)
    real_channel_flow with shape, (3,Nx,Ny,Nz,times)
    returns: the MSE between the xcor of the pred and true'''
    assert tuple(pred_flow.shape)==tuple(real_channel_flow.shape)
    
    pred_xcor = self_xcor(pred_flow)
    true_xcor = self_xcor(real_channel_flow)
    metrics = xcor_metrics(pred_xcor, true_xcor)
    if plot:
        plt.plot(true_xcor,label='true')
        plt.plot(pred_xcor,label='model')
        plt.axvline(np.argmax(pred_xcor), color='orange', linestyle='--', label='model peak')
        plt.axvline(np.argmax(true_xcor), color='blue', linestyle='--', label='true peak')
        plt.legend()
        plt.title('Flow Cross-Correlation: '+title)
        plt.show()
        print('MSE between xcor of pred and true:', metrics['xcor_mse'])
    return metrics

In [ ]:
# test real_channel_flow is "self-consistent" with regards to xcor
cross_correlation_comparison(*real_channel_flow.split(real_channel_flow.shape[-1]//2, dim=-1), title='Real 1st vs 2nd half (consistency check)')

In [ ]:
n_xcor_steps = 25 # NOTE: this should work with the time strides we've tested: 4, 8, and 16
beta = 0.15 # how much to weight the previous MSE vs the new MSE for EMA
plot_interval = max(1, n_xcor_steps//5)

assert sim.flow_thru[-1].shape[-1] == real_channel_flow.shape[-1]
metrics_cum = None # bias-corrected EMA requires direct assignment for the first iteration
for i, end_step in enumerate(np.linspace(0, real_channel_flow.shape[-1], num=n_xcor_steps+1, dtype=int)[1:]):
    i += 1 # 1-based indexing
    should_plot = i % plot_interval == 0 or i == n_xcor_steps
    metrics_i = cross_correlation_comparison(
        sim.flow_thru[-1][...,:end_step],
        real_channel_flow_intermediate[...,:end_step],
        plot=should_plot,
        title=f'$T_0$ vs $T_0+{i/n_xcor_steps}$',
    )
    if metrics_cum is None: metrics_cum = metrics_i.view('f4')
    else: metrics_cum = beta*metrics_cum + (1-beta)*metrics_i.view('f4')
    if should_plot:
        print(f'{end_step=}')
        print('='*75)
metrics_cum = metrics_cum.view(metrics_i.dtype)
print(f'Cummulative metrics with {beta=} and {n_xcor_steps} steps:')
for name in metrics_cum.dtype.names:
    glue_and_print(f'{name}_cum', metrics_cum[name])
for name in metrics_i.dtype.names:
    glue_and_print(f'{name}_last', metrics_i[name])

In [ ]:
# first flow through time
cross_correlation_comparison(sim.flow_thru[0], real_channel_flow_intermediate, title='First flow through time only')

## Getting Samples?

In [ ]:
if sim.uq:
    pred_samples = sim.uq.sample_moments
    pred_samples = [pred_samples_moment[...,-real_channel_flow_intermediate.shape[-1]:] for pred_samples_moment in pred_samples]
    pred_samples = torch.distributions.Normal(*pred_samples).sample()
else: pred_samples = sim.flow_thru[-1][None]
print(f'{real_channel_flow_intermediate.shape=}\n{pred_samples.shape=}')

## Comprehensive Flow Stats Plots:

In [ ]:
'''
This code assumes you have 2 numpy arrays loaded in memory:
    pred_samples with shape, (samples,3,Nx,Ny,Nz,times) (from 1 to 2 flow through times, it can be any flow through time though)
    real_channel_flow with shape, (3,Nx,Ny,Nz,times)
'''

import numpy as np
import matplotlib.pyplot as plt
def E(u):
    uh = np.fft.rfftn(u,axes=[0,2])
    return np.sum(np.abs(uh)**2,axis=-1)
def npmap(f,a):
    return np.array(list(map(f,a)))
def E1d(Lx,Ly,Lz,epsilon_multiplier,nk,u):
    '''
    arguments:
        Lx,Ly,Lz: domain lengths
        epsilon_multiplier: for width of ring to project to point
        nk: number of points along radius to project on to
        u: input function
    output:
        energy spectrum from u. First axis is the energy spectrum. Second is y coordinate
    '''
    nx,ny,nz = u.shape[0:-1]
    kx = np.fft.fftfreq(nx,d=Lx/nx)
    kz = np.fft.rfftfreq(nz,d=Lz/nz)
    dk = np.sqrt(kx[1]**2 + kz[1]**2)
    epsilon = epsilon_multiplier*dk
    Kxz = np.stack(np.meshgrid(kx,kz,indexing='ij'),axis=-1)
    K = np.sqrt(Kxz[...,0]**2 + Kxz[...,1]**2)
    k = np.linspace(0,min(np.max(kx),np.max(kz)),nk)
    Eu = E(u)

    return k,2.*np.transpose(npmap(
        lambda j:npmap(
            lambda ki:np.sum(Eu[:,j][np.abs(K-ki)<epsilon]),
            k),
        np.arange(ny)))

#real_channel_flow.shape==(3,Nx,Ny,Nz,times)
#pred_samples.shape==(samples,3,Nx,Ny,Nz,times)
def plot_1dDiagnostics(pred_samples,real_channel_flow):
    assert pred_samples.shape[1:]==real_channel_flow.shape
    CI_coef = 1.96 # 95% CI

    # Dwyer: I fixed this so it doesn't depend on the number of timesteps (-1 is the last timestep)
    k,EE = E1d(8*np.pi,2.,4*np.pi,epsilon_multiplier=1.0*np.mean(data_module.stride),nk=30,u=torch.swapaxes(real_channel_flow,4,0)[-1])
    Es = []
    for samp in pred_samples: # Dwyer: I fixed this so it doesn't depend on the number of samples
        zzz1 = np.moveaxis(np.asarray(samp[...,-1]),0,-1)
        k,EE1 = E1d(8*np.pi,2.,4*np.pi,epsilon_multiplier=1.0*np.mean(data_module.stride),nk=30,u=zzz1)
        Es.append(EE1)
    Es=np.array(Es)

    trim = 2 # we are trimming the first two because they contain so much energy that they distort the plots
    y_index = real_channel_flow.shape[2]//2 # Dwyer: should be the midpoint b/c it avoids the walls
    mu = np.mean(Es[...,y_index],0)[trim:]
    std = np.std(Es[...,y_index],0)[trim:]
    k,EE = k[trim:],EE[trim:,y_index]

    y = np.loadtxt('y.txt') # Dwyer: we need to interpolate this to the number of y points in the flow
    y = np.interp(np.linspace(0,1,real_channel_flow.shape[2]), np.linspace(0,1,len(y)), y) # interp(x, xp, fp)
    glue_and_print('mse_log_energy_spectrum', MSE(np.log(mu), np.log(EE)))
    fig,ax = plt.subplots(1,4,figsize=(8,2),sharex='col',sharey='col')
    ax[0].loglog(k,EE,'C1') # NOTE: C1 & C2 are colors
    ax[0].loglog(k,mu,'--k')
    ax[0].fill_between(k,mu-CI_coef*std,mu+CI_coef*std)
    ax[0].loglog(k,3e4*k**(-5./3.),'C2') # GOTCHA: this used to use k[2:]!
    #ax[0].set_xscale('linear')
    #ax[0].set_ylim(2e3,5e4)
    ax[0].set_ylabel(r'$E(\kappa)$')
    #ax[0].title.set_text('Energy Spectrum')
    xz_index = 10 # Dwyer: why 10? <-- apparently arbitrary?
    res = pred_samples[:,:,xz_index,:,xz_index,-1].mean(1)
    mu = res.mean(0)
    std = res.std(0)
    true_u1 = real_channel_flow[:,xz_index,:,xz_index,-1].mean(0)
    glue_and_print('mse_bulk_velocity', MSE(mu, true_u1))
    ax[1].plot(y,true_u1,'C1')
    # for i in range(10):
    ax[1].plot(y,mu,'--k')
    ax[1].fill_between(y,mu-CI_coef*std,mu+CI_coef*std)
    ax[1].set_ylabel('$u_{1}$')
    #ax[1].title.set_text('Bulk Velocity')
    res = np.sqrt(pred_samples[:,:,xz_index,:,xz_index,-1].var(1))
    mu = res.mean(0)
    std = res.std(0)
    true_urms = np.sqrt(real_channel_flow[:,xz_index,:,xz_index,-1].var(0))
    glue_and_print('mse_rms', MSE(mu, true_urms))
    ax[2].plot(y,true_urms,'C1')
    # for i in range(10):
    ax[2].plot(y,mu,'--k')
    ax[2].fill_between(y,mu-CI_coef*std,mu+CI_coef*std)
    ax[2].set_ylabel('$u_{rms}$')
    #ax[2].title.set_text('RMS')
    #pred_xcor, true_xcor = cross_correlation_comparison(pred_samples.mean(0),real_channel_flow, plot=False)
    true_xcor = self_xcor(real_channel_flow)
    pred_xcor = [self_xcor(sample) for sample in pred_samples]
    p_xcor_mu = np.mean(pred_xcor,0)
    p_xcor_std = np.std(pred_xcor,0)
    metrics = xcor_metrics(p_xcor_mu, true_xcor)
    for name in metrics.dtype.names:
        glue_and_print(name, metrics[name])
    ax[3].plot(true_xcor,'C1',label='true')
    ax[3].plot(pred_xcor[0],'--k', label='model')
    for i in range(1,len(pred_xcor)):
        ax[3].plot(pred_xcor[i],'--k')
    ax[3].plot(p_xcor_mu,'r',label='mu')
    #ax[3].fill_between(np.arange(len(p_xcor_mu)), p_xcor_mu-CI_coef*p_xcor_std, p_xcor_mu+CI_coef*p_xcor_std)
    #ax[3].title.set_text('Xcor')
    plt.legend(fontsize='xx-small', loc='upper left')
    ax[0].set_xlabel('$\kappa$')
    ax[1].set_xlabel('$y$')
    ax[2].set_xlabel('$y$')
    ax[3].set_xlabel('$x/\Delta x$')
    ax[3].set_ylabel('$u(0) \star u(T)$')
    fig.tight_layout()
    plt.show()

plot_1dDiagnostics(pred_samples, real_channel_flow_intermediate)

# Bayesian Model Validation:

In [ ]:
utils.nvidia_smi(verbose=True)

In [ ]:
import utils
print(f'num model parameters: {utils.count_parameters(model):.5e}')

## Regular Model Metrics with Posterior Averaging
Still valid for MLE models! However MLE models already have these evaluated in W&B and we re-evaluate here to properly evaluate them with BMA.

In [ ]:
data_module.setup() # preload
val_dataloaders = data_module.val_dataloader()
x,y = next(iter(val_dataloaders['long_horizon']))
print(f'{x.shape=}, {y.shape=}')

In [ ]:
L.seed_everything(seed)
device = model.device
assert model.device.type != 'cpu', 'model must be on GPU!'
with PredSamplingWrapper.enable_sampling(25, moments=True):
    trainer = L.Trainer()
    val_metrics = trainer.validate(model=model, dataloaders=val_dataloaders)
    sb.glue('val_metrics', val_metrics)
model = model.to(device) # Trainer keeps moving it back to CPU...

## Posterior-Predictive-Log-Likelihood
(Bayesian only)

In [ ]:
if not isinstance(model, PPOU_NetSimulator):
    from utils import StopExecution
    raise StopExecution

In [ ]:
L.seed_everything(seed)
import model_agnostic_BNN
LPPC_val = model_agnostic_BNN.log_posterior_predictive_check(model, val_dataloaders['val'])
glue_and_print('LPPC_val', LPPC_val)
LPPC_val_long = model_agnostic_BNN.log_posterior_predictive_check(model, val_dataloaders['long_horizon'], n_steps=data_module.long_horizon-1)
glue_and_print('LPPC_val_long', LPPC_val_long)
del val_dataloaders

## Calibration Checks:

In [ ]:
L.seed_everything(seed)
val_loader_small_batch = data_module.val_dataloader(batch_size=2)['val']
cov_calc = model_agnostic_BNN.CoverageCalculator(model, val_loader_small_batch, 15)
del val_loader_small_batch

In [ ]:
cov_calc.plot_truth_quantile_hist()

In [ ]:
approx_ECE = cov_calc.calibration_curve()

In [ ]:
ECE = cov_calc.get_ECE()
sb.glue('ECE', ECE)